# Install and import Required Libraries

In [ ]:
!pip install pandas
!pip install numpy
!pip install seaborn
!pip install matplotlib
!pip install scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
plt.style.use("ggplot")

# Load the dataset

In [ ]:

df = pd.read_csv("../data/yield_df.csv")



In [ ]:
df.head()

# Initial data inspection

In [ ]:
df.shape      #dataset size

In [ ]:
df.columns	 #Column names

In [ ]:
df.info()     #Check data types

In [ ]:
df.describe()     #check statstical summary

In [ ]:
df.describe(include="object")     #for categricaol columns

In [ ]:
df.nunique()    #Check unique values

In [ ]:
df['Area'].unique()       #Cheeck unique Countries

In [ ]:
df['Item'].unique()       #Check unique Items

# Data quality checking

## Checking NULL values

In [ ]:
df.isnull().sum()

In [ ]:
(df.isnull().sum() / len(df)) * 100        #NULL percentage

## Check exact duplicate rows

In [ ]:
int(df.duplicated().sum())

In [ ]:
df[df.duplicated(keep=False)]       #View duplicate values

## Check logical duplicates

In [ ]:
df.duplicated(
    subset=['Area', 'Item', 'Year']
).sum()

In [ ]:
df[
    df.duplicated(
        subset=['Area', 'Item', 'Year'],
        keep=False
    )
]

## Check invalid numerical values

In [ ]:
df[
    (df['hg/ha_yield'] < 0) |
    (df['average_rain_fall_mm_per_year'] < 0) |
    (df['pesticides_tonnes'] < 0)
]

# Data cleaning

## 1) Remove Unnamed: 0 column

In [ ]:
df.drop(columns=['Unnamed: 0'], inplace=True)

In [ ]:
df.head()

## 2) Handle duplicates

In [ ]:
print("Exact duplicate rows:", df.duplicated().sum())

In [ ]:
print(
    "Duplicate Area-Item-Year rows:",
    df.duplicated(
        subset=['Area', 'Item', 'Year']
    ).sum()
)

In [ ]:
duplicate_summary = (
    df[
        df.duplicated(
            subset=['Area', 'Item', 'Year'],
            keep=False
        )
    ]
    .groupby(['Area', 'Item', 'Year'])
    .nunique()
)

duplicate_summary.head(20)

### Remove the 2,310 exact duplicates

In [ ]:
df = df.drop_duplicates()

In [ ]:
df.reset_index(drop=True, inplace=True)

In [ ]:
print("Exact duplicate rows:", df.duplicated().sum())

### Investigate the remaining Area + Item + Year repetitions

In [ ]:
print(
    "Duplicate Area-Item-Year rows:",
    df.duplicated(
        subset=['Area', 'Item', 'Year']
    ).sum()
)

In [ ]:
df[
    (df['Area'] == 'Argentina') &
    (df['Item'] == 'Cassava') &
    (df['Year'] == 1990)
]

In [ ]:
duplicate_summary.max()

### Aggregate the repeated rows

In [ ]:
df = (
    df.groupby(
        ['Area', 'Item', 'Year'],
        as_index=False
    )
    .agg({
        'hg/ha_yield': 'first',
        'average_rain_fall_mm_per_year': 'first',
        'pesticides_tonnes': 'first',
        'avg_temp': 'mean'
    })
)


#first => for Area + Item + Year groups, these columns have a maximum of:1 unique value

In [ ]:
df.reset_index(drop=True, inplace=True)

### Validate after aggregation

In [ ]:
print(
    "Exact duplicate rows:",
    df.duplicated().sum()
)

In [ ]:
print(
    "Duplicate Area-Item-Year rows:",
    df.duplicated(
        subset=['Area', 'Item', 'Year']
    ).sum()
)

In [ ]:
df[
    (df['Area'] == 'Argentina') &
    (df['Item'] == 'Cassava') &
    (df['Year'] == 1990)
]

In [ ]:
df.head()

## 3) Handling Missing values is not need.

## 4) Check data types again

In [ ]:
df.info()

# Exploratory Data Analysis(EDA) and visualization

## 1) Target variable analysis

### Check its statistical summary of the target variable

In [ ]:
df['hg/ha_yield'].describe()

### Visualize the target distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))

sns.histplot(
    df['hg/ha_yield'],
    kde=True
)

plt.title('Distribution of Crop Yield')
plt.xlabel('Crop Yield (hg/ha)')
plt.ylabel('Frequency')

plt.show()

# X-axis — Crop Yield (hg/ha) represents values from your hg/ha_yield column.
# Y-axis — Frequency means how many rows/observations in your dataset have crop yield values within a particular range.

In [ ]:
df['hg/ha_yield'].skew()

In [ ]:
# RESULT:- Right-skewed therefore,there are many records/rows with low crop yield values.

### Check target outliers using a boxplot

In [ ]:
plt.figure(figsize=(8, 4))

sns.boxplot(
    x=df['hg/ha_yield']
)

plt.title('Boxplot of Crop Yield')

plt.show()

In [ ]:
# RESULT:-
#The box shows where most of the main yield values are concentrated.
#The line inside the box is the median (middle yield value).
#The circles/dots after about 215,000 hg/ha are potential outliers (unusually high yield values).

### 2) Country-wise yield analysis

#### Average yield for each country

In [ ]:
country_yield = (
    df.groupby('Area')['hg/ha_yield']
      .mean()
      .sort_values(ascending=False)
)

#### Check the top 10 countries

In [ ]:
country_yield.head(10)

In [ ]:
plt.figure(figsize=(10, 6))

country_yield.head(10).plot(
    kind='bar'
)

plt.title('Top 10 Countries by Average Crop Yield')
plt.xlabel('Country')
plt.ylabel('Average Yield (hg/ha)')
plt.xticks(rotation=45)

plt.show()

#### Check the lowest-yield countries

In [ ]:
country_yield.tail(10)

In [ ]:
plt.figure(figsize=(10, 6))

country_yield.tail(10).plot(
    kind='bar'
)

plt.title('10 Countries with Lowest Average Crop Yield')
plt.xlabel('Country')
plt.ylabel('Average Yield (hg/ha)')
plt.xticks(rotation=45)

plt.show()

## 3) Crop-wise yield analysis

In [ ]:
crop_yield = (
    df.groupby('Item')['hg/ha_yield']
      .mean()
      .sort_values(ascending=False)
)

In [ ]:
crop_yield

In [ ]:
plt.figure(figsize=(12, 6))

crop_yield.plot(
    kind='bar'
)

plt.title('Average Yield by Crop')
plt.xlabel('Crop')
plt.ylabel('Average Yield (hg/ha)')
plt.xticks(rotation=45)

plt.show()

## 4) Year-wise yield analysis

In [ ]:
yearly_yield = (
    df.groupby('Year')['hg/ha_yield']
      .mean()
)

In [ ]:
yearly_yield

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    yearly_yield.index,
    yearly_yield.values,
    marker='o'
)

plt.title('Average Crop Yield Over Time')
plt.xlabel('Year')
plt.ylabel('Average Yield (hg/ha)')

plt.show()

## 5) Numerical feature relationship analysis

### Analyze how numerical input features relate to the target

### Rainfall vs crop yield

In [ ]:
plt.figure(figsize=(8, 5))

sns.scatterplot(
    data=df,
    x='average_rain_fall_mm_per_year',
    y='hg/ha_yield'
)

plt.title('Rainfall vs Crop Yield')
plt.xlabel('Average Rainfall (mm/year)')
plt.ylabel('Crop Yield (hg/ha)')

plt.show()

In [ ]:
#RESULT:- No clear relationship

### Temperature vs crop yield

In [ ]:
plt.figure(figsize=(8, 5))

sns.scatterplot(
    data=df,
    x='avg_temp',
    y='hg/ha_yield'
)

plt.title('Average Temperature vs Crop Yield')
plt.xlabel('Average Temperature')
plt.ylabel('Crop Yield (hg/ha)')

plt.show()

In [ ]:
# RESULT:-
# The points are widely scattered, so this is not a strong clear relationship.
# The scatter plot suggests a possible non-linear relationship between average temperature and crop yield, with higher yields occurring at some moderate temperatures.

### Pesticide usage vs crop yield

In [ ]:
plt.figure(figsize=(8, 5))

sns.scatterplot(
    data=df,
    x='pesticides_tonnes',
    y='hg/ha_yield'
)

plt.title('Pesticide Usage vs Crop Yield')
plt.xlabel('Pesticides (tonnes)')
plt.ylabel('Crop Yield (hg/ha)')

plt.show()

In [ ]:
# RESULT:- NO CLEAR RELATIONSHIP

## 6) Correlation analysis

In [ ]:
# Select numerical values

In [ ]:
numeric_df = df.select_dtypes(
    include=['int64', 'float64']
)

In [ ]:
# Calculate correlation

In [ ]:
correlation_matrix = numeric_df.corr()

correlation_matrix


### Correlation heatmap

In [ ]:
plt.figure(figsize=(8, 6))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm'
)

plt.title('Correlation Matrix of Numerical Features')

plt.show()

In [ ]:
# RESULT:- Considering the target variable, all numerical features show weak linear correlations with crop yield.

## 7) Crop-specific relationships

### Temperature vs Crop Yield by Crop Type

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x='avg_temp',
    y='hg/ha_yield',
    hue='Item'
)

plt.title('Temperature vs Crop Yield by Crop Type')

plt.show()

### Analyze crops seperately

#### Maize

In [ ]:
maize_df = df[df['Item'] == 'Maize']

In [ ]:
plt.figure(figsize=(8, 5))

sns.scatterplot(
    data=maize_df,
    x='avg_temp',
    y='hg/ha_yield'
)

plt.title('Temperature vs Maize Yield')

plt.show()

In [ ]:
sns.scatterplot(
    data=maize_df,
    x='average_rain_fall_mm_per_year',
    y='hg/ha_yield'
)

plt.title('Rainfall vs Maize Yield')

plt.show()

#### Potatoes

In [ ]:
potatoes_df = df[df['Item'] == 'Potatoes']

In [ ]:
plt.figure(figsize=(8, 5))

sns.scatterplot(
    data=potatoes_df,
    x='avg_temp',
    y='hg/ha_yield'
)

plt.title('Temperature vs Potatoes Yield')

plt.show()

In [ ]:
sns.scatterplot(
    data=potatoes_df,
    x='average_rain_fall_mm_per_year',
    y='hg/ha_yield'
)

plt.title('Rainfall vs Potatoes Yield')

plt.show()

In [ ]:
df.head()

# Feature and Target Selection

### 1) Select features (X)

In [ ]:
X = df.drop(columns=['hg/ha_yield'])

### 2) Select Target (y)

In [ ]:
y = df['hg/ha_yield']

### 3) Check X and y

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
print("X shape:", X.shape)
print("y shape:", y.shape)

# Train-Test Split

### 1) Import train_test_split

In [ ]:
from sklearn.model_selection import train_test_split

### 2) Split X and y

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
X_train

In [ ]:
y_train

In [ ]:
X_test

In [ ]:
y_test

### 3) Check the shapes

In [ ]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
print("Training data:", len(X_train))
print("Testing data:", len(X_test))

# Data Preprocessing

## 1) Define categorical and numerical columns

In [ ]:
categorical_features = [
    'Area',
    'Item'
]

numerical_features = [
    'Year',
    'average_rain_fall_mm_per_year',
    'pesticides_tonnes',
    'avg_temp'
]

## 2) Import preprocessing tools

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 3) Create the preprocessor

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'cat',
            OneHotEncoder(handle_unknown='ignore'),
            categorical_features
        ),
        (
            'num',
            StandardScaler(),
            numerical_features
        )
    ]
)

# Build a Baseline Model Using a Pipeline

### 1)Import Pipeline and Linear Regression

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

### 2) Create the pipeline

In [ ]:
baseline_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', LinearRegression())
    ]
)

### 3) Train the baseline model



In [ ]:
baseline_pipeline.fit(X_train, y_train)

### 4) Make predictions

In [ ]:
y_pred_baseline = baseline_pipeline.predict(X_test)

### 5) Compare actual and predicted values

In [ ]:
comparison = pd.DataFrame({
    'Actual Yield': y_test.values,
    'Predicted Yield': y_pred_baseline
})

In [ ]:
comparison.head(10)

# Evaluate the Baseline Model Using MAE, RMSE, and R² Score

### 1) Import evaluation metrics

In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
import numpy as np

### 2) Calculate MAE

In [ ]:
mae = mean_absolute_error(
    y_test,
    y_pred_baseline
)

### 3) Calculate RMSE

In [ ]:
rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_baseline
    )
)

### 4) Calculate R² Score

In [ ]:
r2 = r2_score(
    y_test,
    y_pred_baseline
)

### 5) Baseline results

In [ ]:
print("Baseline Model: Linear Regression")
print("MAE:", mae)
print("RMSE:", rmse)
print("R² Score:", r2)

In [ ]:
### Visualize Actual vs Predicted Yield

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

plt.scatter(
    y_test,
    y_pred_baseline,
    alpha=0.5
)

plt.xlabel("Actual Yield")
plt.ylabel("Predicted Yield")
plt.title("Actual vs Predicted Yield - Linear Regression")

plt.show()

#### Conclution according to results of the evaluation metrics

In [ ]:
#MAE = 28,612.72
#On average, the predicted crop yield differs from the actual crop yield by approximately 28,613 hg/ha.

#RMSE = 41,270.89
#The RMSE is higher than the MAE, suggesting that the model has some relatively large prediction errors.

#R² = 0.7352
#The model explains approximately 73.5% of the variance in crop yield.

# Train and Compare Multiple Regression Models

### 1) Import the regression models

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

### 2) Create a dictionary of models

In [ ]:
models = {
    'Linear Regression': LinearRegression(),

    'Decision Tree': DecisionTreeRegressor(
        random_state=42
    ),

    'Random Forest': RandomForestRegressor(
        random_state=42,
        n_jobs=-1
    ),

    'Gradient Boosting': GradientBoostingRegressor(
        random_state=42
    )
}

### 3) Train and evaluate each model

In [ ]:
results = []

for name, model in models.items():

    pipeline = Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            ('model', model)
        ]
    )

    # Train model
    pipeline.fit(X_train, y_train)

    # Make predictions
    y_pred = pipeline.predict(X_test)

    # Calculate evaluation metrics
    mae = mean_absolute_error(
        y_test,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            y_pred
        )
    )

    r2 = r2_score(
        y_test,
        y_pred
    )

    # Store results
    results.append({
        'Model': name,
        'MAE': mae,
        'RMSE': rmse,
        'R2 Score': r2
    })

    print(name, "completed")

### 4) Comparison table inclusing

In [ ]:
results_df = pd.DataFrame(results)
results_df

### 5) Sort models by R² Score

In [ ]:
results_df = results_df.sort_values(
    by='R2 Score',
    ascending=False
)

results_df

# Model Selection and Validation

### 1) Create the Random Forest pipeline

In [ ]:
rf_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        (
            'model',
            RandomForestRegressor(
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

### 2) Perform cross-validation

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    rf_pipeline,
    X_train,
    y_train,
    cv=5,
    scoring='r2',
    n_jobs=-1
)


### 3) Display cross-validation results

In [ ]:
print("Cross-validation R² scores:", cv_scores)

print(
    "Mean CV R² Score:",
    cv_scores.mean()
)

print(
    "CV Standard Deviation:",
    cv_scores.std()
)

### 4) Check training vs testing performance

#### train the Random Forest pipeline

In [ ]:
rf_pipeline.fit(
    X_train,
    y_train
)

#### Calculate training R²

In [ ]:
train_r2 = rf_pipeline.score(
    X_train,
    y_train
)

#### Calculate testing R²

In [ ]:
test_r2 = rf_pipeline.score(
    X_test,
    y_test
)

In [ ]:
print("Training R² Score:", train_r2)
print("Testing R² Score:", test_r2)

In [ ]:
# Since R² scores are approximately very close to each other,
#    The Random Forest model performs consistently across the five cross-validation folds.

# Hyperparameter Tuning of the Random Forest Model

### 1) Import RandomizedSearchCV

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

### 2) Define the hyperparameter search space

In [ ]:
param_distributions = {
    'model__n_estimators': [100, 200, 300, 500],

    'model__max_depth': [
        None,
        10,
        20,
        30,
        40
    ],

    'model__min_samples_split': [
        2,
        5,
        10
    ],

    'model__min_samples_leaf': [
        1,
        2,
        4
    ],

    'model__max_features': [
        1.0,
        'sqrt',
        'log2'
    ]
}

### 3) Create RandomizedSearchCV

In [ ]:
random_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_distributions,
    n_iter=50,
    cv=5,
    scoring='r2',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

### 4) Run hyperparameter tuning

In [ ]:
random_search.fit(
    X_train,
    y_train
)

### 5) Display the best hyperparameters

In [ ]:
print(
    "Best Hyperparameters:"
)

print(
    random_search.best_params_
)

### 6) Check the best cross-validation score

In [ ]:
print(
    "Best CV R² Score:",
    random_search.best_score_
)

### 7) Get the best tuned model

In [ ]:
best_rf_model = random_search.best_estimator_

In [ ]:
print(best_rf_model)

Immediately save the tuned model

In [ ]:
import joblib

joblib.dump(
    best_rf_model,
    "../models/crop_yield_model.pkl"
)

Check that it was saved

In [ ]:
import os

os.path.exists(
    "../models/crop_yield_model.pkl"
)